In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results" / "vlm"

# Load spatial locations and image metadata
locations = gpd.read_file(
    PROCESSED / "panorama_locations.geojson"
)

metadata = pd.read_csv(
    PROCESSED / "perspective_views_metadata.csv"
)

# Load the two prediction sets
original = pd.read_csv(
    RESULTS / "vlm_predictions.csv"
)

evidence = pd.read_csv(
    RESULTS / "evidence_first_all.csv"
)

print("Locations:", len(locations))
print("Image metadata:", len(metadata))
print("Original predictions:", len(original))
print("Evidence-first predictions:", len(evidence))

In [ ]:
hybrid = original[
    [
        "image_id",
        "greenery",
        "pedestrian",
        "cycling",
        "motor_vehicle"
    ]
].merge(
    evidence[
        [
            "image_id",
            "cycling",
            "cycling_evidence"
        ]
    ].rename(
        columns={
            "cycling": "cycling_revised"
        }
    ),
    on="image_id",
    validate="one_to_one"
)

hybrid["cycling"] = hybrid["cycling_revised"]

hybrid = hybrid.drop(columns=["cycling_revised"])

hybrid = hybrid.merge(
    metadata[["image_id", "location_id", "filename"]],
    on="image_id",
    validate="one_to_one"
)

assert len(hybrid) == 100
assert hybrid["location_id"].nunique() == 50

display(hybrid.head())

In [ ]:
# Check coordinate system and available columns
print("CRS:", locations.crs)
print("Location columns:", locations.columns.tolist())

# Join each image to its panorama location
spatial = locations.merge(
    hybrid,
    on="location_id",
    how="inner",
    validate="one_to_many"
)

# Export in standard longitude/latitude coordinates
spatial = spatial.to_crs("EPSG:4326")

assert len(spatial) == 100
assert spatial["location_id"].nunique() == 50
assert spatial["image_id"].is_unique
assert spatial.geometry.notna().all()

print("Spatial records:", len(spatial))
print("Unique locations:", spatial["location_id"].nunique())
print("CRS:", spatial.crs)

display(
    spatial[
        [
            "location_id", "image_id",
            "greenery", "pedestrian",
            "cycling", "motor_vehicle"
        ]
    ].head()
)

In [ ]:
OUTPUT = PROCESSED / "streetscape_hybrid.geojson"

spatial.to_file(OUTPUT, driver="GeoJSON")

# Read it back to verify the exported file
check = gpd.read_file(OUTPUT)

assert len(check) == 100
assert check["location_id"].nunique() == 50
assert check["image_id"].is_unique

print("Saved:", OUTPUT)
print("Features:", len(check))
print("Unique locations:", check["location_id"].nunique())

In [ ]:
OUTPUT = PROCESSED / "streetscape_hybrid.geojson"

spatial = gpd.read_file(OUTPUT)

In [ ]:
import folium
from IPython.display import display

# One marker per location, with both views in its popup
location_groups = spatial.groupby("location_id", sort=True)

center = [
    spatial.geometry.y.mean(),
    spatial.geometry.x.mean()
]

m = folium.Map(
    location=center,
    zoom_start=13,
    tiles="OpenStreetMap"
)

for location_id, group in location_groups:
    point = group.geometry.iloc[0]

    view_sections = []

    for _, view in group.sort_values("image_id").iterrows():
        evidence_text = str(view["cycling_evidence"])

        section = f"""
        <div style="margin-bottom:14px">
            <h4 style="margin-bottom:5px">{view['image_id']}</h4>
            <b>Greenery:</b> {view['greenery']}<br>
            <b>Pedestrian:</b> {view['pedestrian']}<br>
            <b>Cycling:</b> {view['cycling']}<br>
            <b>Motor vehicle:</b> {view['motor_vehicle']}<br>
            <p><b>Cycling evidence:</b> {evidence_text}</p>
        </div>
        """
        view_sections.append(section)

    popup_html = f"""
    <div style="width:340px;max-height:420px;overflow-y:auto">
        <h3>{location_id}</h3>
        {''.join(view_sections)}
    </div>
    """

    folium.CircleMarker(
        location=[point.y, point.x],
        radius=6,
        color="#1764a0",
        fill=True,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=380),
        tooltip=location_id
    ).add_to(m)

display(m)

In [ ]:
WEB = ROOT / "web"
WEB.mkdir(exist_ok=True)

m.save(str(WEB / "streetscape_map_preview.html"))